# 12 · Remainder fibers and hidden carries

Can motion reveal the kernel of a map, and can the kernel's size drive another construction?

We fold integers by two remainders, measure the fibers, separate coincident occurrences,
and cycle each fiber using a **measured** period. Then we copy one period to rebuild the
set—and discover why addition on the resulting layers still needs a carry.

Try predicting the pictures for `(a,b)=(7,5)` and `(6,4)` before changing the parameters.
The default deliberately breaks coprimality. Neither `gcd` nor `lcm` supplies our definitions.
Those functions appear only in independent checks. All parameters here are positive integers.

The [lesson and from-blank canvas guide](../docs/lessons/12_residue_fibers.md) connects this
investigation to quotient groups and the generalized Chinese remainder theorem.

In [ ]:
from pathlib import Path
import json
import math
import sys
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Video
from kaleion import Collection, F, Inspection, Workspace, param
from kaleion.comparison import compare_keyed_values
from kaleion.viewers.plotly import snapshot_figure, transition_figure, animation_figure
from kaleion.viewers.video import write_mp4

ROOT = Path.cwd() if (Path.cwd() / "src/kaleion").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks"))
from examples.studio.coverage import unique_assignment
from lesson_views import style, save_figures
from snapshot_views import rectangular_values
OUT = ROOT / "build/notebooks/12_residue_fibers"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES = {}
pio.renderers.default = "plotly_mimetype+notebook"
PARAMETERS = {"a": 6, "b": 4}
assert all(type(x) is int and x > 0 for x in PARAMETERS.values())
a, b = param("a"), param("b")

## 1. A map and its kernel

Let $N=ab$ and $f(n)=(n\bmod a,n\bmod b)$ for $0\le n<N$.
Its kernel is the subset sent to $(0,0)$. Measure its size $d$, then propose a period $L=N/d$.
We will justify this division below; for now it is an inspectable candidate.

In [ ]:
integers = (Collection.grid(a*b, axes=("n",), values=F.n)
            .annotate(r=F.n % a, s=F.n % b).arrange(F.n, 0, 0))
kernel = integers.where((F.r == 0) & (F.s == 0))
kernel_size = kernel.count()
period = kernel_size.with_values(a*b // F.value)
d, L = kernel_size.bind(on=0), period.bind(on=0)
triples = Collection.grid(a, b, a*b, axes=("r", "s", "n"), values=F.n)
relation = triples.where((F.n % a == F.r) & (F.n % b == F.s))
counts = relation.count(by=(F.r, F.s)).arrange(F.r, F.s, F.value)
pairs = Collection.grid(a, b, axes=("r", "s"), values=1).arrange(F.r, F.s, 0)
compatible = pairs.where((F.r - F.s) % d == 0)
membership = compatible.count(by=(F.r, F.s)).arrange(F.r, F.s, 0)
predicted = membership.with_values(d * F.value)
workspace = Workspace({"Integers": integers, "Kernel": kernel, "Kernel size": kernel_size,
                       "Period": period, "Relation": relation, "Counts": counts,
                       "Pairs": pairs, "Prediction": predicted}, PARAMETERS, max_items=2000)
assert not workspace.state.errors, dict(workspace.state.errors)
print("Kernel size:", workspace.state.results["Kernel size"].values.tolist())
print("Measured period:", workspace.state.results["Period"].values.tolist())

## 2. Give zeros a domain

We deliberately included **every** pair $(r,s)$ in the witness cube. Counting its $n$ axis
keeps empty fibers. Reducing only the observed images would lose the unvisited pairs.
The gray/zero cells below are present measurements, not missing data.

At $(6,4)$, there are twelve fibers of size two and twelve of size zero. At $(7,5)$,
all thirty-five have size one. The total number of witnesses is always $ab$.

In [ ]:
state = workspace.state
xs, ys, rows = rectangular_values(state.results["Counts"], x="s", y="r")
matrix = np.asarray(rows, dtype=int)
print(matrix)
count_figure = style(go.Figure(go.Heatmap(
    z=matrix.tolist(), x=list(range(PARAMETERS["b"])), y=list(range(PARAMETERS["a"])),
    text=matrix.tolist(), texttemplate="%{text}", colorscale="Viridis", zmin=0,
    hovertemplate="r=%{y}, s=%{x}<br>fiber size=%{z}<extra></extra>")),
    "Every remainder pair has a measured fiber")
count_figure.update_xaxes(title="s = n mod b", dtick=1)
count_figure.update_yaxes(title="r = n mod a", dtick=1)
FIGURES["fiber-counts"] = count_figure
count_figure.show()
domain = [(r, s) for r in range(PARAMETERS["a"]) for s in range(PARAMETERS["b"])]
naive = compare_keyed_values(state.results["Counts"], state.results["Pairs"],
                            left_keys=("r", "s"), right_keys=("r", "s"), domain=domain)
corrected = compare_keyed_values(state.results["Counts"], state.results["Prediction"],
                                left_keys=("r", "s"), right_keys=("r", "s"), domain=domain)
assert naive.same_domain and corrected.holds
assert naive.holds == (math.gcd(*PARAMETERS.values()) == 1)
print("All-ones claim:", naive.holds, "Numerical discrepancies:", len(naive.nonzero))
assert sum(state.results["Counts"].values) == PARAMETERS["a"] * PARAMETERS["b"]

## 3. Separate coincident occurrences using order

First put every integer at its remainder pair. Coincidence does not identify occurrences.
Next order **within each fiber** by $n$ and measure strict ranks. A keyed read of that
rank supplies the height; we did not hardcode the heights as $\lfloor n/L\rfloor$.

In [ ]:
ranks = integers.group_by(F.r, F.s).order_by(F.n).ranks(key=F.n)
workspace.set("Ranks", ranks)
folded = integers.arrange(F.r, F.s, 0)
workspace.set("Moving", folded)
stacked = integers.arrange(F.r, F.s, ranks.bind(on=F.n))
separation = workspace.set("Moving", stacked)
FIGURES["separate-fibers"] = transition_figure(separation, "Moving", steps=31,
    title="Measured ranks separate the same occurrences")
FIGURES["separate-fibers"].show()
assert set(separation.before.results["Moving"].ids) == set(separation.after.results["Moving"].ids)
length = int(workspace.state.results["Period"].values[0])
assert workspace.state.results["Ranks"].values.tolist() == [n//length for n in range(PARAMETERS["a"]*PARAMETERS["b"])]

Inspect a top-layer point. Its placement reads a measured rank, whose contributors
are the earlier occurrences in that fiber. Also inspect the zero-pair fiber; at `(6,4)`
its witnesses are `0` and `12`. For the coprime case the selected point has rank zero.

In [ ]:
inspector = Inspection(workspace.state)
probe_n = min(length + 1, PARAMETERS["a"]*PARAMETERS["b"] - 1)
read = inspector.bindings(inspector.find("Moving", probe_n, by=("n",)))[0]
rank_receipt = inspector.measurement(read.driver)
fiber_receipt = inspector.measurement(inspector.find("Counts", (0,0), by=("r","s")))
print("Rank:", rank_receipt.item.value, "Earlier integers:", [c.item.value for c in rank_receipt.contributors])
print("Zero-pair witnesses:", [c.item.fields["n"] for c in fiber_receipt.contributors])
receipts = {"rank": rank_receipt.to_dict(), "zero_fiber": fiber_receipt.to_dict()}
if math.gcd(*PARAMETERS.values()) > 1:
    zero = inspector.measurement(inspector.find("Counts", (0,1), by=("r","s")))
    assert zero.item.value == 0 and zero.contributor_count == 0
    receipts["empty_fiber"] = zero.to_dict()

## 4. Let the measured period act

Roll the original $n$ slots by $L$. The integer label stays with its occurrence; the
logical $n$ field now names the destination slot. At $(6,4)$ this exchanges the two
layers while preserving both residues. The moving points pass through intermediate
screen coordinates; only the endpoints describe the group action.

This is translation by an element of the kernel, a permutation **within** every fiber.
For coprime moduli the kernel has one element and the action is stationary.

In [ ]:
cycled = stacked.roll(axis="n", shift=L)
cycle = workspace.set("Moving", cycled)
reverse = workspace.undo()
forward_samples = [cycle.frame("Moving", float(t)) for t in np.linspace(0,1,25)]
reverse_samples = [reverse.frame("Moving", float(t)) for t in np.linspace(0,1,25)]
for before, after in zip(forward_samples, reversed(reverse_samples)):
    np.testing.assert_allclose(before.positions, after.positions, atol=1e-12, rtol=0)
workspace.redo()
FIGURES["kernel-action"] = animation_figure(forward_samples + reverse_samples,
    labels=["Add the measured period"]*25 + ["Undo the captured path"]*25,
    title="Kernel translation keeps each remainder pair fixed")
FIGURES["kernel-action"].show()

## 5. Copy a period: a set decomposition

Select $0\le n<L$, order those representatives, and repeat them $d$ times.
`scalar()` explicitly requires one value for the repeat count. It is different from a
keyed read per target item. Each copy gets a new identity and keeps its captured parent.
The carried `n` field remains its representative; the new label is $n+L\cdot\mathrm{sheet}$.

In [ ]:
first = integers.where(F.n < L).select().order_by(F.n)
copied = first.tile(kernel_size.scalar()).annotate(sheet=F.index // L)
lifted = copied.with_values(F.n + L*F.sheet).arrange(F.r, F.s, F.sheet)
workspace.set("Copied", lifted)
copies = workspace.state.results["Copied"]
assert copies.values.tolist() == list(range(PARAMETERS["a"]*PARAMETERS["b"]))
assert set(copies.ids).isdisjoint(workspace.state.results["Integers"].ids)
FIGURES["copied-sheets"] = snapshot_figure(copies, title="A new copy for each representative and sheet")
FIGURES["copied-sheets"].update_traces(marker_color=["#49c5b6" if int(h)%2==0 else "#f0bc63" for h in copies.fields["sheet"]])
FIGURES["copied-sheets"].update_layout(scene_xaxis_title="r", scene_yaxis_title="s", scene_zaxis_title="Sheet")
FIGURES["copied-sheets"].show()

## 6. The hidden carry: set coordinates are not componentwise group coordinates

Write $n=t+Lh$ with $0\le t<L$ and $0\le h<d$. Addition modulo $ab=Ld$ becomes

$$ (t,h)+(u,k)=\left((t+u)\bmod L,\;\left(h+k+\left\lfloor\frac{t+u}{L}\right\rfloor\right)\bmod d\right). $$

At $(6,4)$, the pair `11 + 1` crosses from representative `11` on sheet `0` to
representative `0` on sheet `1`. Ignoring the carry predicts `0`; the correct result is `12`.
The carry is an example of the extra data needed to describe a **group extension**.
The copied layers alone do not declare a group isomorphism.

In [ ]:
addition = Collection.grid(a*b, a*b, values=(F.i+F.j) % (a*b))
t, u, h, k = F.i % L, F.j % L, F.i // L, F.j // L
without_carry = addition.with_values((t+u) % L + L*((h+k) % d))
with_carry = addition.with_values((t+u) % L + L*((h+k+(t+u)//L) % d))
sums = Workspace({"Sum": addition, "No carry": without_carry, "With carry": with_carry}, PARAMETERS)
assert not sums.state.errors
np.testing.assert_array_equal(sums.state.results["Sum"].values, sums.state.results["With carry"].values)
residual = sums.state.results["Sum"].values - sums.state.results["No carry"].values
print("Missing-carry discrepancies:", int(np.count_nonzero(residual)))
if PARAMETERS == {"a":6,"b":4}:
    assert int(residual[11*24+1]) == 12
    assert np.count_nonzero(residual) == 264
N = PARAMETERS["a"]*PARAMETERS["b"]
FIGURES["hidden-carry"] = style(go.Figure(go.Heatmap(
    z=np.asarray(residual, dtype=int).reshape(N,N).tolist(), colorscale="RdBu", zmid=0,
    hovertemplate="left=%{y}, right=%{x}<br>omitted-carry residual=%{z}<extra></extra>")),
    "Where componentwise addition predicts the wrong integer")
FIGURES["hidden-carry"].show()

## 7. An inverse is a claim with assumptions

Use the existing coverage recipe to propose exactly one integer for **every** pair.
It fails at `(6,4)` while the relation and counts remain available. Changing to `(7,5)`
makes it a valid inverse over all pairs. Selecting one representative from each occupied
fiber would be a different construction with a different expected domain.

In [ ]:
inverse = unique_assignment(relation, ["r","s"], pairs, ["r","s"], F.n, "address")
claim = Workspace({"Relation": relation, "Inverse": inverse, "Counts": counts}, {"a":6,"b":4})
assert set(claim.state.errors) == {"Inverse"}
print("Noncoprime inverse:", claim.state.errors["Inverse"])
claim.set_parameters(a=7, b=5)
assert not claim.state.errors
assert sorted(claim.state.results["Inverse"].fields["address"].tolist()) == list(range(35))
print("Coprime inverse: all 35 addresses occur once")

## 8. Why the pattern holds

Put $g=\gcd(a,b)$, write $a=gA,b=gB$ with $\gcd(A,B)=1$, and let
$M=gAB=\operatorname{lcm}(a,b)$. An integer has both remainders zero exactly when
it is a multiple of $M$. There are $g$ such integers in $[0,ab)$, so our measured
$d=g$ and $L=ab/d=M$.

Two integers have the same pair of remainders exactly when their difference is a
multiple of $L$. Thus each occupied fiber has the form
$t,t+L,\ldots,t+(d-1)L$ for a unique $0\le t<L$. This proves its size, its ranks,
the copy recipe, and the kernel translation.

A pair $(r,s)$ is occupied exactly when $r\equiv s\pmod d$. Necessity follows by
subtracting the two congruences. For sufficiency solve
$Aq\equiv(s-r)/d\pmod B$, possible because $A$ is invertible modulo $B$;
then $r+aq$ has the desired residues. When $B=1$ the congruence imposes no restriction.
This proves the predicted count formula and explains the coprime CRT as the case $d=1$.

The homomorphism $f$ identifies the quotient
$\mathbb Z/(ab)\,/\ker f$ with the compatible residue pairs, also with $\mathbb Z/L$.
More precisely those pairs form the fiber product
$\mathbb Z/a\times_{\mathbb Z/d}\mathbb Z/b$: both coordinates must agree after reduction modulo $d$.

The additive exact sequence
$0\to\mathbb Z/d\xrightarrow{h\mapsto Lh}\mathbb Z/(ab)\xrightarrow{n\mapsto n\bmod L}\mathbb Z/L\to0$
describes the layers. For $d>1$ it does not split: $d\mid L$, so every element of
$\mathbb Z/L\times\mathbb Z/d$ is annihilated by $L$, but `1` in $\mathbb Z/(Ld)$ is not.
The carry formula follows directly by expanding the integer sum. These arguments,
not the plotted cases, establish the general statements.

Background: [J. S. Milne, *Algebraic Number Theory*, §1, Chinese remainder theorem](https://www.jmilne.org/math/CourseNotes/ANTc.pdf).
Our elementary fiber and carry arguments specialize the integer situation; no proof assistant is involved.

## 9. Export and replay a two-dimensional chart

For video, place the representatives on the horizontal axis and their measured ranks
on the vertical axis. This is an explicit second chart, not geometry fed back into the
mathematics. The 3D views above remain interactive in their exported HTML files.
The movie unfolds the line into sheets, applies the measured shift, then undoes both.

In [ ]:
flat = integers.arrange(F.n, 0)
sheet_chart = integers.arrange(F.n % L, ranks.bind(on=F.n))
movie = Workspace({"Chart": flat}, PARAMETERS)
unfold = movie.set("Chart", sheet_chart)
turn = movie.set("Chart", sheet_chart.roll(axis="n", shift=L))
transitions = [unfold, turn, movie.undo(), movie.undo()]
captions = ["Unfold into measured sheets", "Add the measured period", "Undo the shift", "Undo the unfolding"]
samples = [transition.frame("Chart", float(t)) for transition in transitions for t in np.linspace(0,1,25)]
labels = [caption for caption in captions for _ in range(25)]
video = write_mp4(samples, OUT / "residue-sheets.mp4", labels=labels,
                  title="Remainder fibers and a measured kernel action", fps=20)
display(Video(str(video), embed=True, width=900))
save_figures(OUT, FIGURES)
(OUT / "workspace.json").write_text(workspace.to_json())
(OUT / "movie.json").write_text(movie.to_json())
(OUT / "evidence.json").write_text(json.dumps({"parameters": PARAMETERS, "receipts": receipts,
    "naive": naive.to_dict(), "corrected": corrected.to_dict(),
    "carry_discrepancies": int(np.count_nonzero(residual))}, indent=2))
reopened = Workspace.from_json((OUT / "workspace.json").read_text())
assert reopened.state.results["Copied"].values.tolist() == copies.values.tolist()
print("Wrote", len(FIGURES), "interactive figures, 100 video frames, captured workspaces and evidence to", OUT)

## Further experiments

- Change to `(6,6)`, `(1,5)`, or `(1,1)`. Predict the kernel, empty fibers, stationary motion, and carry before rerunning.
- Roll by `1` instead of `L`. Follow the original integer label and identify the step that changes the sheet.
- Change the source domain to a proper prefix of `[0,ab)`. Uniform fibers can fail: the domain assumption matters as well as coprimality.
- Next: observe several commuting shifts, reconstruct their orbit lattice, and investigate Smith normal form. A diagonal matrix should emerge from declared integer shears, rather than a dedicated “solve” button.

The explicit witness product costs $(ab)^2$ occurrences. The workspace here limits each
intermediate to 2,000 items (so try $ab\le44$). The primitive recipes remain independent of
the viewer. Scalar constructor reads are recorded in the graph; the current pointwise
binding inspector does not yet supply a scalar-read receipt. Inspect `Kernel size` directly.